# Results Analysis — READ-ONLY (no model runs)

Loads `results/metrics/all_candidates.csv` and `backbone_summary.csv`. Performs quality checks, generates figures, and writes `results/analysis_summary.md`. Reports failures separately. If no real data, prints clear message and stops.


In [1]:
# Imports + load saved CSVs. No model computation.
import os, sys, json, pathlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

# Try project repo (Colab-mounted) or fall back to local CWD
for candidate_root in [Path("/content/rso-protein-design-exploration"), Path.cwd()]:
    if (candidate_root / "results" / "metrics").exists():
        PROJECT = candidate_root
        break
else:
    PROJECT = Path.cwd()
print(f"Using PROJECT = {PROJECT}")

RESULTS = PROJECT / "results"
METRICS = RESULTS / "metrics"
FIGURES = PROJECT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

CAND_CSV = METRICS / "all_candidates.csv"
BB_CSV   = METRICS / "backbone_summary.csv"

if not CAND_CSV.exists():
    print("=" * 72)
    print("No completed experimental runs found.")
    print(f"Expected CSV at: {CAND_CSV}")
    print("")
    print("Run notebook 01_rso_reproduction.ipynb and/or 02_length_experiment.ipynb")
    print("first, then run scripts/collect_metrics.py to populate results/metrics/.")
    print("=" * 72)
    raise SystemExit(0)

cand = pd.read_csv(CAND_CSV)
print(f"Loaded all_candidates.csv: {len(cand)} rows × {len(cand.columns)} cols")
bb = pd.read_csv(BB_CSV) if BB_CSV.exists() else None
if bb is not None:
    print(f"Loaded backbone_summary.csv: {len(bb)} rows × {len(bb.columns)} cols")


Using PROJECT = /Users/shenyz/projects/github/rso-protein-design-exploration
Loaded all_candidates.csv: 8 rows × 32 cols
Loaded backbone_summary.csv: 1 rows × 11 cols


In [2]:
# Schema + consistency validation

def validate_candidates_schema(df):
    required = {"backbone_id", "candidate_id", "sequence", "rmsd_angstrom", "tm_score", "mean_plddt"}
    missing = required - set(df.columns)
    if missing:
        return [f"Missing required columns: {sorted(missing)}"]
    errs = []
    if "length" in df.columns and "sequence" in df.columns:
        bad_len = df[df["sequence"].str.len() != df["length"]]
        if not bad_len.empty:
            errs.append(f"{len(bad_len)} row(s) have sequence length != declared length")
    for col in ["rmsd_angstrom", "tm_score", "mean_plddt"]:
        if col in df.columns:
            null_count = df[col].isna().sum()
            if null_count:
                errs.append(f"{null_count} NULL value(s) in column '{col}'")
    return errs

schema_errors = validate_candidates_schema(cand)
if schema_errors:
    print("SCHEMA ERRORS:")
    for e in schema_errors: print(f"  - {e}")
else:
    print("Schema validation: PASS")

# Per-row sequence-length match
if "length" in cand.columns and "sequence" in cand.columns:
    mismatches = cand.loc[cand["sequence"].str.len() != cand["length"],
                          ["backbone_id", "candidate_id", "length"]]
    mismatches["actual_len"] = cand.loc[mismatches.index, "sequence"].str.len()
    if not mismatches.empty:
        print("\nSequence-length mismatches:")
        display(mismatches)

# Failed candidates (separate JSONs with 'error' key)
failed_rows = []
RUNS = RESULTS / "runs"
if RUNS.exists():
    for run_dir in sorted(RUNS.iterdir()):
        if not run_dir.is_dir(): continue
        s3 = run_dir / "stage3_validation"
        if not s3.is_dir(): continue
        for m in sorted(s3.glob("*_metrics.json")):
            try:
                data = json.loads(m.read_text())
                if "error" in data:
                    failed_rows.append({
                        "run": run_dir.name,
                        "file": m.name,
                        "error": data["error"],
                        "runtime_s": data.get("validation_runtime_seconds"),
                    })
            except: pass
if failed_rows:
    print(f"\nFailed validation candidates ({len(failed_rows)}):")
    display(pd.DataFrame(failed_rows))
else:
    print("\nNo failed candidates detected in per-candidate JSONs.")


Schema validation: PASS

No failed candidates detected in per-candidate JSONs.


In [3]:
# Ranking table per backbone

rank_cols_present = all(c in cand.columns for c in ["backbone_id", "rmsd_angstrom", "tm_score", "mean_plddt"])

if rank_cols_present and len(cand) > 0:
    ranked_frames = []
    for bb_id, grp in cand.groupby("backbone_id"):
        g = grp.copy()
        g["rank_rmsd"]  = g["rmsd_angstrom"].rank(ascending=True,  method="min").astype("Int64")
        g["rank_tm"]    = g["tm_score"].rank(ascending=False, method="min").astype("Int64")
        g["rank_plddt"] = g["mean_plddt"].rank(ascending=False, method="min").astype("Int64")
        ranked_frames.append(g.sort_values("rank_rmsd"))
    ranked = pd.concat(ranked_frames, ignore_index=True)
    out_cols = [c for c in ["backbone_id", "candidate_id", "length", "rmsd_angstrom",
                            "tm_score", "mean_plddt", "rank_rmsd", "rank_tm", "rank_plddt"]
                if c in ranked.columns]
    print("Per-backbone ranked candidates (first 20 rows):")
    display(ranked[out_cols].head(20))
    ranking_csv = METRICS / "per_backbone_ranked.csv"
    ranked.to_csv(ranking_csv, index=False)
    print(f"Full ranking written -> {ranking_csv}")
else:
    print("Insufficient columns to produce per-backbone ranking.")


Per-backbone ranked candidates (first 20 rows):


,backbone_id,candidate_id,length,rmsd_angstrom,tm_score,mean_plddt,rank_rmsd,rank_tm,rank_plddt
0,bb0,c0,100,0.508548,0.9816,94.28,1,1,1
1,bb0,c5,100,0.612442,0.9740,91.60,2,2,8
2,bb0,c4,100,0.666189,0.9728,92.56,3,3,4
3,bb0,c2,100,0.692886,0.9692,94.26,4,4,2
4,bb0,c1,100,0.741296,0.9657,93.43,5,5,3
5,bb0,c6,100,0.783221,0.9606,92.34,6,6,6
6,bb0,c3,100,0.849906,0.9552,92.33,7,7,7
7,bb0,c7,100,0.871818,0.9534,92.55,8,8,5


Full ranking written -> /Users/shenyz/projects/github/rso-protein-design-exploration/results/metrics/per_backbone_ranked.csv


In [4]:
# Generate figures — via project script if available, else inline

fig_script = PROJECT / "scripts" / "make_figures.py"
written_figures = []

if fig_script.exists():
    print(f"Invoking {fig_script} ...")
    r = subprocess.run([sys.executable, str(fig_script)], capture_output=True, text=True)
    print(r.stdout)
    if r.stderr.strip(): print("STDERR:", r.stderr)
    written_figures = sorted(str(p) for p in FIGURES.glob("*"))
else:
    print("scripts/make_figures.py not found; generating standard figures inline.")
    df = cand.copy()

    def _save(fig, name):
        out = FIGURES / name
        fig.savefig(out, dpi=150, bbox_inches="tight")
        plt.close(fig)
        written_figures.append(str(out))

    if all(c in df.columns for c in ["rmsd_angstrom", "tm_score", "mean_plddt"]):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].hist(df["rmsd_angstrom"].dropna(), bins=20, color="#4C72B0", edgecolor="white")
        axes[0].set_xlabel("RMSD (A)"); axes[0].set_ylabel("Count")
        axes[0].set_title("RMSD distribution")
        axes[1].hist(df["tm_score"].dropna(), bins=20, color="#55A868", edgecolor="white")
        axes[1].set_xlabel("TM-score"); axes[1].set_ylabel("Count")
        axes[1].set_title("TM-score distribution")
        axes[2].hist(df["mean_plddt"].dropna(), bins=20, color="#C44E52", edgecolor="white")
        axes[2].set_xlabel("mean pLDDT"); axes[2].set_ylabel("Count")
        axes[2].set_title("mean pLDDT distribution")
        fig.tight_layout()
        _save(fig, "metric_histograms.png")

    if all(c in df.columns for c in ["length", "rmsd_angstrom"]) and df["length"].nunique() > 1:
        fig, ax = plt.subplots(figsize=(7, 4))
        df.boxplot(column="rmsd_angstrom", by="length", ax=ax, grid=False)
        ax.set_xlabel("Protein length (aa)"); ax.set_ylabel("RMSD (A)")
        fig.suptitle(""); ax.set_title("RMSD by length")
        fig.tight_layout()
        _save(fig, "rmsd_by_length_box.png")

    if all(c in df.columns for c in ["length", "tm_score"]) and df["length"].nunique() > 1:
        fig, ax = plt.subplots(figsize=(7, 4))
        df.boxplot(column="tm_score", by="length", ax=ax, grid=False)
        ax.set_xlabel("Protein length (aa)"); ax.set_ylabel("TM-score")
        fig.suptitle(""); ax.set_title("TM-score by length")
        fig.tight_layout()
        _save(fig, "tm_by_length_box.png")

    if all(c in df.columns for c in ["length", "mean_plddt"]) and df["length"].nunique() > 1:
        fig, ax = plt.subplots(figsize=(7, 4))
        df.boxplot(column="mean_plddt", by="length", ax=ax, grid=False)
        ax.set_xlabel("Protein length (aa)"); ax.set_ylabel("mean pLDDT")
        fig.suptitle(""); ax.set_title("mean pLDDT by length")
        fig.tight_layout()
        _save(fig, "plddt_by_length_box.png")

print(f"\nWritten figures ({len(written_figures)} file(s)):")
for f in written_figures: print(f"  - {f}")


Invoking /Users/shenyz/projects/github/rso-protein-design-exploration/scripts/make_figures.py ...


[info] Candidates loaded: 8
[info] Numeric value counts: {'rmsd_angstrom': 8, 'tm_score': 8, 'mean_plddt': 8, 'total_runtime_seconds': 8}
[info] Loss histories loaded: 1
  wrote /Users/shenyz/projects/github/rso-protein-design-exploration/figures/rmsd_vs_length.png
  wrote /Users/shenyz/projects/github/rso-protein-design-exploration/figures/tmscore_vs_length.png
  wrote /Users/shenyz/projects/github/rso-protein-design-exploration/figures/plddt_vs_length.png
  wrote /Users/shenyz/projects/github/rso-protein-design-exploration/figures/runtime_vs_length.png
  wrote /Users/shenyz/projects/github/rso-protein-design-exploration/figures/optimization_loss_100aa.png
  wrote /Users/shenyz/projects/github/rso-protein-design-exploration/figures/candidate_metrics_100aa.png
  wrote /Users/shenyz/projects/github/rso-protein-design-exploration/figures/structure_overlay.png

STDERR: /Users/shenyz/projects/github/rso-protein-design-exploration/src/rso_exploration/plotting.py:101: FutureWarning: 

The `c

In [5]:
# Per-length descriptive stats + box plots display

if "length" in cand.columns and cand["length"].nunique() > 1:
    num_cols = [c for c in ["rmsd_angstrom", "tm_score", "mean_plddt", "ptm"] if c in cand.columns]
    desc = cand.groupby("length")[num_cols].describe().round(3)
    print("Per-length descriptive statistics:")
    display(desc)
    desc_csv = METRICS / "per_length_descriptive.csv"
    desc.to_csv(desc_csv)
    print(f"Written -> {desc_csv}")

    # Show the per-length box plots in-notebook (inline-generated)
    fig, axes = plt.subplots(1, len(num_cols), figsize=(5 * len(num_cols), 4))
    if len(num_cols) == 1: axes = [axes]
    for ax, col in zip(axes, num_cols):
        cand.boxplot(column=col, by="length", ax=ax, grid=False)
        ax.set_xlabel("Protein length (aa)"); ax.set_ylabel(col)
        fig.suptitle("")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print("No length column / only one length — skipping per-length boxplots.")


No length column / only one length — skipping per-length boxplots.


In [6]:
# Write results/analysis_summary.md

lines = []
lines.append("# RSO Protein Design — Analysis Summary")
lines.append("")
lines.append("Generated by notebooks/03_results_analysis.ipynb.")
lines.append(f"- Generated at: {pd.Timestamp.utcnow().isoformat()}Z")
lines.append("")

n_candidates = len(cand)
n_backbones = cand["backbone_id"].nunique() if "backbone_id" in cand.columns else None
lines.append("## Sample size")
lines.append(f"- Total candidates: **{n_candidates}**")
if n_backbones is not None:
    lines.append(f"- Unique backbones:  **{n_backbones}**")
if "length" in cand.columns:
    lens = sorted(cand["length"].unique().tolist())
    lines.append(f"- Lengths evaluated: {lens}")
lines.append("")

def _fmt(v, unit="", higher_better=False):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "N/A"
    sign = "↑" if higher_better else "↓"
    return f"{float(v):.3f}{unit} {sign}"

if "rmsd_angstrom" in cand.columns:
    r_best = cand["rmsd_angstrom"].dropna().min() if cand["rmsd_angstrom"].notna().any() else None
    r_worst = cand["rmsd_angstrom"].dropna().max() if cand["rmsd_angstrom"].notna().any() else None
    lines.append("## RMSD (Cα, Å)")
    lines.append(f"- Best:  {_fmt(r_best,  unit=' Å', higher_better=False)}")
    lines.append(f"- Worst: {_fmt(r_worst, unit=' Å', higher_better=False)}")
    if r_best is not None:
        lines.append(f"- Median: {cand['rmsd_angstrom'].dropna().median():.3f} Å")
    lines.append("")

if "tm_score" in cand.columns:
    t_best  = cand["tm_score"].dropna().max() if cand["tm_score"].notna().any() else None
    t_worst = cand["tm_score"].dropna().min() if cand["tm_score"].notna().any() else None
    lines.append("## TM-score (0..1)")
    lines.append(f"- Best:  {_fmt(t_best,  higher_better=True)}")
    lines.append(f"- Worst: {_fmt(t_worst, higher_better=True)}")
    if t_best is not None:
        lines.append(f"- Median: {cand['tm_score'].dropna().median():.3f}")
    lines.append("")

if "mean_plddt" in cand.columns:
    p_best  = cand["mean_plddt"].dropna().max() if cand["mean_plddt"].notna().any() else None
    p_worst = cand["mean_plddt"].dropna().min() if cand["mean_plddt"].notna().any() else None
    lines.append("## mean pLDDT (0..100)")
    lines.append(f"- Best:  {_fmt(p_best,  higher_better=True)}")
    lines.append(f"- Worst: {_fmt(p_worst, higher_better=True)}")
    if p_best is not None:
        lines.append(f"- Median: {cand['mean_plddt'].dropna().median():.3f}")
    lines.append("")

if failed_rows:
    lines.append("## Failures")
    lines.append(f"- {len(failed_rows)} candidate validation(s) recorded an 'error' field. ")
    lines.append(f"  See notebook output above for list.")
    lines.append("")

lines.append("## Notes")
lines.append("")
if n_candidates == 0 or n_candidates < 24:
    lines.append("- **Small n — no statistical claims.** Repeat notebook 02 with more seeds/lengths before drawing conclusions.")
else:
    lines.append("- Sample size supports qualitative trends; statistical significance not assessed.")
if bb is None:
    lines.append("- backbone_summary.csv not present — backbone-level provenance not summarized here.")

out_md = RESULTS / "analysis_summary.md"
out_md.write_text("\n".join(lines))
print(f"Written -> {out_md}")
print("")
print("--- Preview ---")
print(out_md.read_text())


Written -> /Users/shenyz/projects/github/rso-protein-design-exploration/results/analysis_summary.md

--- Preview ---
# RSO Protein Design — Analysis Summary

Generated by notebooks/03_results_analysis.ipynb.
- Generated at: 2026-09-16T15:04:37.728669+00:00Z

## Sample size
- Total candidates: **8**
- Unique backbones:  **1**
- Lengths evaluated: [100]

## RMSD (Cα, Å)
- Best:  0.509 Å ↓
- Worst: 0.872 Å ↓
- Median: 0.717 Å

## TM-score (0..1)
- Best:  0.982 ↑
- Worst: 0.953 ↑
- Median: 0.967

## mean pLDDT (0..100)
- Best:  94.280 ↑
- Worst: 91.600 ↑
- Median: 92.555

## Notes

- **Small n — no statistical claims.** Repeat notebook 02 with more seeds/lengths before drawing conclusions.
